In [1]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn import pipeline
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

In [2]:
df = (
    pd.read_csv('train.csv')
    .pipe(lambda x: x[['PassengerId', 'Survived', 'Pclass', 'Sex', 'Age',
                       'SibSp', 'Parch', 'Fare', 'Cabin', 'Ticket','Embarked']])
    .pipe(lambda x: x.set_index('PassengerId'))
    .pipe(lambda x: x.assign(Ticket_code = x['Ticket'].str.split(' ').str[0] ))
    .pipe(lambda x: x.assign(Cabin = x['Cabin'].str[0] ))
    .pipe(lambda x: x.assign(Sex = [True if i == 'male' else False for i in x['Sex'] ]))
    .pipe(lambda x: x.drop(columns='Ticket'))
    .pipe(lambda x: x.assign(FamilySize = x['SibSp'] + x['Parch'] + 1))
    .pipe(lambda x: x.drop(columns=['SibSp', 'Parch']))
)

In [3]:
X = df.drop('Survived', axis=1)
y = df['Survived']

In [4]:
x_train, x_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y
)

In [5]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

In [6]:
numerical = [
    'Pclass',
    'Sex',
    'Age',
    'Fare',
    'FamilySize',
]
categorical = [
    'Cabin',
    'Ticket_code',
    'Embarked'
]

In [7]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            'num',
            SimpleImputer(strategy='mean'),
            numerical
        ),
        (
            'cat',
            Pipeline([
              ('imputer', SimpleImputer(strategy='most_frequent')),
              ('encoder', OneHotEncoder(handle_unknown='ignore'))
            ]),
            categorical
        )
    ]
)

In [8]:
model = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', DecisionTreeClassifier(
        max_depth=5,
        random_state=9
    ))
])

In [9]:
model.fit(x_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](8,)","['Pclass','Sex','Age',...,'Embarked','Ticket_code','FamilySize']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,8
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all rema

In [10]:
y_pred = model.predict(x_test)

In [11]:
print(accuracy_score(y_pred,y_test))

0.8268156424581006


In [12]:
model.fit(x_train, y_train)
print("Train:", model.score(x_train, y_train))
print("Test :", model.score(x_test, y_test))

Train: 0.8651685393258427
Test : 0.8268156424581006


In [13]:
for depth in range(1, 100):
    model = Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', DecisionTreeClassifier(
            max_depth= 5,
            random_state= 9
        ))
    ])

    model.fit(x_train, y_train)

    print(
        depth,
        model.score(x_train, y_train),
        model.score(x_test, y_test)
    )

1 0.8651685393258427 0.8268156424581006
2 0.8651685393258427 0.8268156424581006
3 0.8651685393258427 0.8268156424581006
4 0.8651685393258427 0.8268156424581006
5 0.8651685393258427 0.8268156424581006
6 0.8651685393258427 0.8268156424581006
7 0.8651685393258427 0.8268156424581006
8 0.8651685393258427 0.8268156424581006
9 0.8651685393258427 0.8268156424581006
10 0.8651685393258427 0.8268156424581006
11 0.8651685393258427 0.8268156424581006
12 0.8651685393258427 0.8268156424581006
13 0.8651685393258427 0.8268156424581006
14 0.8651685393258427 0.8268156424581006
15 0.8651685393258427 0.8268156424581006
16 0.8651685393258427 0.8268156424581006
17 0.8651685393258427 0.8268156424581006
18 0.8651685393258427 0.8268156424581006
19 0.8651685393258427 0.8268156424581006
20 0.8651685393258427 0.8268156424581006
21 0.8651685393258427 0.8268156424581006
22 0.8651685393258427 0.8268156424581006
23 0.8651685393258427 0.8268156424581006
24 0.8651685393258427 0.8268156424581006
25 0.8651685393258427 0.8

In [14]:
from sklearn.model_selection import cross_val_score

scores = cross_val_score(
    model,
    X,
    y,
    cv=5,
    scoring='accuracy'
)

print(scores)
print("Mean:", scores.mean())

[0.82681564 0.81460674 0.82022472 0.79775281 0.85393258]
Mean: 0.822666499278137


In [15]:
for depth in range(1, 40):

    model = Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', DecisionTreeClassifier(
            max_depth=depth,
            random_state=42
        ))
    ])

    scores = cross_val_score(
        model,
        X,
        y,
        cv=5,
        scoring='accuracy'
    )

    print(depth, scores.mean())

1 0.7867365513778168
2 0.773316176009039
3 0.8080660347749671
4 0.8069926558282594
5 0.8204193082669009
6 0.8237712635741635
7 0.8226539451384094
8 0.8338773460548616
9 0.8215240725629277
10 0.8372418555018518
11 0.8316364321134895
12 0.828278199736363
13 0.8417425145941874
14 0.8316364321134895
15 0.8383780051471973
16 0.8361245370660976
17 0.832772581758835
18 0.8361433682756889
19 0.8327788588286987
20 0.8361308141359614
21 0.8383654510074697
22 0.8361308141359614
23 0.8350134957002071
24 0.8383654510074697
25 0.8383717280773336
26 0.8383717280773336
27 0.8383717280773336
28 0.8350134957002071
29 0.8383654510074697
30 0.8338961772644529
31 0.8316489862532169
32 0.8350072186303432
33 0.8338961772644529
34 0.8316489862532169
35 0.8316489862532169
36 0.830525390747599
37 0.830525390747599
38 0.8316489862532169
39 0.8316489862532169


In [16]:
from sklearn.ensemble import RandomForestClassifier

model = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(
        n_estimators=100,
        max_depth=15,
        random_state=42
    ))
])

In [17]:
model.fit(x_train,y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](8,)","['Pclass','Sex','Age',...,'Embarked','Ticket_code','FamilySize']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,8
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all rema

In [18]:
y_pred = model.predict(x_test)
print(accuracy_score(y_pred,y_test))

0.770949720670391


In [19]:
accuracy_score( y_test,y_pred)

0.770949720670391

In [20]:
for n in [10,30, 50,60,70,80,90, 100]:
    model = Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', RandomForestClassifier(
            n_estimators=n,
            max_depth=5,
            random_state=9
        ))
    ])

    scores = cross_val_score(
        model,
        X,
        y,
        cv=5,
        scoring='accuracy'
    )

    print(n, scores.mean())

10 0.7385474860335195
30 0.7329357855752934
50 0.7407883999748917
60 0.7475362500784635
70 0.7497708869499717
80 0.7520180779612078
90 0.7553700332684704
100 0.7497771640198355


In [21]:
for depth in [3, 5, 7, 10, 15, 20, None]:

    model = Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', RandomForestClassifier(
            n_estimators=50,
            max_depth=depth,
            random_state=42
        ))
    ])

    scores = cross_val_score(
        model,
        X,
        y,
        cv=5,
        scoring='accuracy'
    )

    print(depth, scores.mean())

3 0.6913941372167473
5 0.7284539576925491
7 0.7643964597325967
10 0.7857259431297471
15 0.8171364007281401
20 0.8294645659406189
None 0.8384093904965162


In [22]:
for leaf in [1, 2, 4, 8, 12]:

    model = Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', RandomForestClassifier(
            n_estimators=50,
            max_depth=None,
            min_samples_leaf=leaf,
            random_state=42
        ))
    ])

    scores = cross_val_score(
        model,
        X,
        y,
        cv=5,
        scoring='accuracy'
    )

    print(leaf, scores.mean())

1 0.8384093904965162
2 0.7497771640198356
4 0.668941058313979
8 0.6184106459104891
12 0.617287050404871


In [23]:
for i in range(1,10):

    model = Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', RandomForestClassifier(
            n_estimators=50,
            max_depth=None,
            max_features=0.5,
            min_samples_split=9,
            min_samples_leaf=i,
            random_state=42
        ))
    ])

    scores = cross_val_score(
        model,
        X,
        y,
        cv=5,
        scoring='accuracy'
    )

    print(i, scores.mean())

1 0.8406440273680247
2 0.8350386039796623
3 0.8339150084740442
4 0.8294331805913
5 0.8305630531667818
6 0.8260812252840374
7 0.8204569706860838
8 0.8260812252840374
9 0.8125980792166217


In [28]:
final_model = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(
        n_estimators=50,
        max_depth=None,
        max_features=0.5,
        min_samples_split=9,
        min_samples_leaf=1,
        random_state=42
    ))
])

final_model.fit(x_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](8,)","['Pclass','Sex','Age',...,'Embarked','Ticket_code','FamilySize']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,8
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all rema

In [29]:
y_pred = final_model.predict(x_test)

print(accuracy_score(y_test, y_pred))

0.8156424581005587


In [30]:
from sklearn.metrics import confusion_matrix, classification_report
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

[[98 12]
 [21 48]]
              precision    recall  f1-score   support

           0       0.82      0.89      0.86       110
           1       0.80      0.70      0.74        69

    accuracy                           0.82       179
   macro avg       0.81      0.79      0.80       179
weighted avg       0.81      0.82      0.81       179



## Why you're stuck at ~80% (and why 95% is the wrong goal)

1. **Your depth sweep does nothing.** The loop loops over `depth` but the model is hardcoded to `max_depth=5` — the variable is never used, so all 99 iterations train the *identical* model. You never actually tuned anything:

   ```python
   for depth in range(1, 100):
       ...
       DecisionTreeClassifier(max_depth=5, random_state=9)   # depth never used!
   ```

2. **The next sweep crashed with `NameError: name 'cross_val_score' is not defined`** because `cross_val_score` was imported in a cell *after* the one that calls it. Jupyter runs cells in execution order, not top-to-bottom — re-run the whole notebook from the top.

3. **95% is not reachable on Titanic with honest evaluation.** The dataset caps honest models at ~80-85% (only ~890 rows, ~20% of labels are unreliable). Low-hundreds 'solutions' that beat that are leakage or overfit to the leaderboard. Your 83-84% CV is already near the realistic ceiling — good tuning buys you a point or two, not fifteen.

In [27]:
from sklearn.model_selection import GridSearchCV, cross_val_score
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier

rf = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42))
])

param_grid = {
    'classifier__n_estimators': [100, 200, 400],
    'classifier__max_depth': [None, 10, 20],
    'classifier__min_samples_leaf': [1, 2, 4],
    'classifier__max_features': [None, 0.3, 0.5, 0.7],
}

grid = GridSearchCV(rf, param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid.fit(x_train, y_train)

print('Best CV accuracy:', grid.best_score_)
print('Best params:', grid.best_params_)

y_pred = grid.best_estimator_.predict(x_test)
print('Held-out test accuracy:', accuracy_score(y_test, y_pred))

hgb = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', HistGradientBoostingClassifier(random_state=42, max_iter=200))
])
scores = cross_val_score(hgb, X, y, cv=5, scoring='accuracy')
print('HistGradientBoosting CV:', scores.mean())

Best CV accuracy: 0.8441051905840637
Best params: {'classifier__max_depth': 10, 'classifier__max_features': 0.7, 'classifier__min_samples_leaf': 1, 'classifier__n_estimators': 400}
Held-out test accuracy: 0.8100558659217877


ValueError: 
All the 5 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
5 fits failed with the following error:
Traceback (most recent call last):
  File "C:\Users\user\PycharmProjects\titanicKaggle\.venv\Lib\site-packages\sklearn\model_selection\_validation.py", line 860, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
    ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\user\PycharmProjects\titanicKaggle\.venv\Lib\site-packages\sklearn\base.py", line 1403, in wrapper
    return fit_method(estimator, *args, **kwargs)
  File "C:\Users\user\PycharmProjects\titanicKaggle\.venv\Lib\site-packages\sklearn\pipeline.py", line 645, in fit
    self._final_estimator.fit(Xt, y, **last_step_params["fit"])
    ~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^